## Import modules

In [ ]:
import numpy as np
import pandas as pd
from reactionmodel.model import Species
from reactionmodel.hook import HookAwareModel
import reactionmodel.parser as parser

## Load arrows from the excel document

This notebook reads a spreadsheet defining all the transitions between states in our model, and turns it into a `Model` object that can be used in Gillespie simulation.

This notebook requires the two packages I have written:
- [reactionmodel](https://github.com/tanderson11/reactionmodel): to create the `Model` object, which represents a set of `Species` (compartments) and `Reactions` (arrows between compartments). The `Model` knows how to produce the stoichiometry matrix, the rate constant matrix, the propensity vector, and the derivate `dydt`.
- [hybrid](https://github.com/tanderson11/hybrid/tree/main): to simulate a `Model` using either Gillespie simulation, $tau$-leaping, or Haseltine-Rawlings hybrid partitioning.

If you don't want to use `hybrid` to simulate the model, it may still be useful to use `reactionmodel` to load the model from the spreadsheet. Since a `reactionmodel.Model` can easily export information like the stoichiometry matrix, it should be easy to use with whatever forward simulator you are interested in. Each of these two packages has enough documentation in their README to get you up and running.

### How does this package work with the spreadsheet?

This package expects an Excel document with four different sheets:

- **Species**: a table of information about the different compartments of the model. It must have the following columns:
  - Name: a label for this species/compartment.
  - PRESSURE_DS: the relative amount of colonization pressure for drug sensitive bacteria that an individual in this compartment exerts on their contacts. (Ex. 0, for someone not colonized by DS, but $\psi_\text{colonization}$ for someone with a low level colonization with DS.)
  - PRESSURE_DR: the same as PRESSURE_DS but for DR bacteria.
- **Reactions**: a table of arrows between compartments. It must have the following columns:
  - Group: a label for what *kind* of transition this is (ex. "Admission"). This label is used for what I call "triggered" reactions. Because we want to model a fixed population size in a healthcare environment where new patients are admitted when existing ones die/are discharged, we make it so that every arrow with the group label "Discharge" triggers one of the arrows with label "Admission."
  - Include: a T/F column (we use checkboxes) where the value True means that this transition will be included in the model. Useful for quickly turning features on/off without deleting anything.
  - Description: a description of the arrow in words.
  - Description stem: a prefix to the description, this is used as a cosmetic tool in the spreadsheet to allow someone to visually scan and see which arrows have something but not everything in common (because they have the same stem but a different description).
  - Initial (A): together with "Initial (N)", defines the compartment where this arrow starts. (A = abundance, N = niche). A value of "*" means "make one of these arrows for all possible states" (a * can be used for A, for N, or for both). A value of "-" for A and "-" for N means this arrow does not start in compartment but produces a new individual (ex. Admission).
  - Initial (N): see above.
  - Final (A): Like "Initial (A)" but for where the arrow ends.
  - Final (N): See above.
  - trigger: If this is not blank, it should be filled with a valid "Group" label. This transition does not occur except when one of the transitions in the named group occurs. In this case, the rate of the reaction is interpreted as a probability that this reaction is the one within its group that occurs when the trigger happens.
  - k: The rate constant, written as a mathematical expression in terms of the parameters of the model.
  - k stem: A factor multiplied with "k" to produce the rate constant. Like "Description stem", this is just for the convenience of being able to visually scan for similarities and differences between the different arrows.
- **Parameters**: a table of model parameters.
  - Parameter: the name of the parameter (ex. "beta" for the probability of infection from a potentially infectious encounter with a contact).
  - Value: the value of the parameter as a number.
- **Hooks**: a table defining which transitions trigger other transitions.
  - Triggering group: when a transition with this "Group" label takes place (see **Reactions**), an event from the "Triggered group" will take place.
  - Triggered group: a transition from this group will take place immediately after a transition from "Triggering group" takes place. The triggered transition will be chosen at random from all the events in the triggered group, with probability defined under the column "k" in **Reactions**.

### What is nice about this approach?

- Parameter names are not coupled to parameter values. The model transitions are defined entirely in **Reactions**, and their rates are written entirely in terms of the different model parameters. But then **Parameters** sheet encodes the *values* of those parameters. This means that once you are happy with the model transitions you have written down in **Reactions**, you don't have to touch them while you iterate on different parametrizations of the model using whatever data / guesses best suit your exploration.

**CAVEAT**: At the moment, I only know for certain that Gillespie simulation will work with the triggered reactions. I will need to check the other simulation methods if they are called for.

In [ ]:
def concatenate_columns(df, columns, parentheses=True, sep='*', na_rep=None):
    df = df[columns].copy()
    if parentheses:
        df['left'] = '('
        df['right'] = ')'
    else:
        df['left'] = ''
        df['right'] = ''
    for c in columns:
        df[c] = df['left'].str.cat(df[c]).str.cat(df['right'])
    
    first_col = df[columns[0]]
    later_cols = df[columns[1:]]
    return first_col.str.cat(later_cols, sep=sep, na_rep=na_rep)

In [ ]:
arrows_file = pd.ExcelFile('./Shared model arrows.xlsx')
arrows = arrows_file.parse('Reactions').drop(0)
arrows = arrows[arrows['Include'].astype(bool)]
arrows['Initial'] = arrows['Initial (A)'].str.cat(arrows['Initial (N)'], sep='_')
arrows['Final'] = arrows['Final (A)'].str.cat(arrows['Final (N)'], sep='_')

arrows['Initial'] = arrows['Initial'].str.replace('-_-', '')
arrows['Final'] = arrows['Final'].str.replace('-_-', '')
# It's dumb, but k stem is guaranteed non-blank where Description (ie not Description stem) is guaranteed non-blank
arrows['k'] = concatenate_columns(arrows, ['k stem', 'k']).fillna(arrows['k stem'])
arrows['Description'] = concatenate_columns(arrows, ['Description stem', 'Description'], sep=' ', parentheses=False).fillna(arrows['Description'])

arrows = arrows[['Group', 'Description', 'Initial', 'Final', 'k', 'trigger']]

arrows

In [ ]:
hooks = arrows_file.parse('Hooks').set_index('Triggering group')
hooks

## Parse file into a Model

In [ ]:
# PROBLEM: DSDR can't exist in high abundance
families = {
    'star': ['M', 'X', 'DS', 'DR', 'DSDR']
}

In [ ]:
def parse_arrow(arrow):
    family_flag = False

    if pd.isna(arrow.trigger):
        arrow_dictionary = {'k': arrow.k}
    else:
        arrow_dictionary = {'p': arrow.k}

    reactants = arrow.Initial
    if reactants.count('*') > 1: assert False
    elif reactants.count('*') == 1: family_flag = True
    reactants = reactants.replace('*', '$i')

    products = arrow.Final
    if products.count('*') > 1: assert False
    elif products.count('*') == 1: family_flag = True
    products = products.replace('*', '$i')

    reactants = [reactants] if reactants else []
    products = [products] if products else []

    arrow_dictionary.update({
        'description': arrow.Description,
        'reactants': reactants,
        'products': products,
    })

    if arrow.Group in hooks.index:
        arrow_dictionary['hooked_set'] = hooks.loc[arrow.Group]['Triggered group']

    if family_flag:
        arrow_dictionary['used_families'] = {'i': 'star'}

    return arrow_dictionary

In [ ]:
from collections import defaultdict
reactions = []
hooked_sets = defaultdict(lambda: [])

# collect triggered reactions first
for i, arrow in arrows.iterrows():
    if pd.isna(arrow.trigger):
        continue

    arrow_dictionary = parse_arrow(arrow)
    hooked_sets[arrow.Group].append(arrow_dictionary)


for i, arrow in arrows.iterrows():
    if not pd.isna(arrow.trigger):
        continue

    arrow_dictionary = parse_arrow(arrow)
    reactions.append(arrow_dictionary)

In [ ]:
species_df = arrows_file.parse('Species')
base_species = [Species(s.Name, description=s.Description) for _,s in species_df.fillna('').iterrows()]

bad_species = [
    {'name':'DS_M'},
    {'name':'DS_X'},
    {'name':'DS_DR'},
    {'name':'DR_M'},
    {'name':'DR_X'},
    {'name':'DR_DS'},
    {'name':'X_M'},
    {'name':'DSDR_DSDR'},
    {'name':'DSDR_DS'},
    {'name':'DSDR_DR'},
]
bad_species = [Species(**s) for s in bad_species]

species = [s.to_dict() for s in base_species] + [s.to_dict() for s in bad_species]

In [ ]:
model_dictionary = {
    'families': families,
    'species': species,
    'reactions': reactions,
    'triggered_sets': hooked_sets
}

results = parser.loads(model_dictionary, model_class=HookAwareModel)

In [ ]:
filtered_reactions = [r for r in results.model.all_reactions if not set(r.reactants).intersection(bad_species) and not set(r.products).intersection(bad_species)]

In [ ]:
filtered_model = HookAwareModel(
    base_species,
    [r for r in results.model.all_reactions if not set(r.reactants).intersection(bad_species) and not set(r.products).intersection(bad_species)]
)

In [ ]:
parameters = arrows_file.parse('Parameters')[['Parameter', 'Value']].set_index('Parameter').to_dict()['Value']
## At some point here evaluate the rate constants at the parameters

needs_pressure_ds = [i for i,r in enumerate(filtered_model.all_reactions) if 'PRESSURE_DS' in r.k]
needs_pressure_dr = [i for i,r in enumerate(filtered_model.all_reactions) if 'PRESSURE_DR' in r.k]

parameters.update({'PRESSURE_DS': 1, 'PRESSURE_DR': 1})

all_k = filtered_model.get_k(parameters=parameters)
assert(isinstance(all_k, np.ndarray))

In [ ]:
import numpy as np
from reactionmodel.model import eval_expression

s_df = species_df.set_index('Name')
pressure_ds_vec = np.zeros(len(filtered_model.species))
pressure_dr_vec = np.zeros(len(filtered_model.species))

for i,s in enumerate(filtered_model.species):
    pressure_ds_vec[i] = eval_expression(str(s_df.loc[s.name]['PRESSURE_DS']), parameters)
    pressure_dr_vec[i] = eval_expression(str(s_df.loc[s.name]['PRESSURE_DR']), parameters)

# Install parameters (colonization pressure defined here!)

In [ ]:
N = 250
parameters['beta'] = parameters['beta'] / N

In the **Species** sheet, we marked which compartments exert colonization pressure on susceptible compartments. But we want to leave open different options for how we determine the *total* colonization pressure on an individual. For example, maybe we want there to be a zeroth order rate defining the pressure from a contaminated environment.

Below, we define two functions of $f(t,y)$ that return the colonization pressure from DS and DR bacteria, respectively. These functions have access to the current state $y$, which is how we implement mass action kinetics.

By default, **all** we do is mass action kinetics, with relative rates of contact as defined in **Species**. But you could easily modify these functions by programming whatever you want to model!

In [ ]:
from numba import jit as numbajit

@numbajit(nopython=True)
def pressure_ds(t, y):
    pressure = np.zeros(1)
    pressure[0] = np.sum(y * pressure_ds_vec)
    return pressure

@numbajit(nopython=True)
def pressure_dr(t, y):
    pressure = np.zeros(1)
    pressure[0] = np.sum(y * pressure_dr_vec)
    return pressure

In [ ]:
parametrized_reactions = []
for i,rxn in enumerate(filtered_model.all_reactions):
    k = all_k[i]
    if i in needs_pressure_ds:
        assert i not in needs_pressure_dr
        k = pressure_ds
    elif i in needs_pressure_dr:
        assert i not in needs_pressure_ds
        k = pressure_dr
    existing_dict = rxn.to_dict(keep_species_objects=True)
    existing_dict.update({'k': k})
    parametrized_reactions.append(type(rxn)(**existing_dict))

In [ ]:
np.sum(np.zeros(len(filtered_model.species)) * pressure_dr_vec, keepdims=True)

In [ ]:
parametrized_reactions = [r for r in parametrized_reactions if r.k != 0]

In [ ]:
parametrized_model = HookAwareModel(
    base_species,
    parametrized_reactions
)

# Model

Now we have loaded a `Model` object and given it parameters, so the model is truly defined and ready for simulation.

In [ ]:
parametrized_model

# Gillespie forward simulation

In [ ]:
# These are helper functions to create plots from our model.
# Because we have lots of states, its helpful to group them together under broader labels
# Such as "susceptible infection" for anyone in DS_DSDR or DS_DS.

i_to_partition = {}
for i,label in enumerate(parametrized_model.legend()):
    if 'DS_' in label:
        partition = 'susceptible infection'
    elif 'DR_' in label:
        partition = 'resistant infection'
    elif '_DSDR' in label:
        partition = 'dual colonization'
    elif '_DS' in label:
        partition = 'DS colonization'
    elif '_DR' in label:
        partition = 'DR colonization'
    else:
        partition = 'uninfected uncolonized'
    i_to_partition[i] = partition

def sum_and_relabel(m, state):
    partition_to_amount = {}
    for i,x in enumerate(state):
        try:
            partition_to_amount[i_to_partition[i]] += x
        except KeyError:
            partition_to_amount[i_to_partition[i]] = x
    
    return partition_to_amount

In [ ]:
from hybrid.gillespie import GillespieSimulator
s = GillespieSimulator.from_model(parametrized_model, parameters=parameters, jit=False)

In [ ]:
initial = parametrized_model.make_initial_condition({'M_M': N*0.8, 'M_DS': N*0.15, 'M_DR': N*0.05})

In [ ]:
result = s.simulate([0, 1.0], initial, np.random.default_rng())

In [ ]:
# Use the sum_and_relabel tool on point in the history
new_y = []
for y in result.y_history.T:
    new_y.append(sum_and_relabel(parametrized_model, y))

# Make a dataframe of the results
# A Pandas DataFrame is designed to match R dataframes, so hopefully it will be somewhat familiar 
df = pd.DataFrame(new_y, index=result.t_history)

(df / N).plot()